# 🏸 BMCA - Badminton Match Coaching Assistant

Upload a badminton match video and get AI-powered coaching insights.

**Steps:**
1. Run Setup cell (installs dependencies + clones repo)
2. Upload your video
3. Run the pipeline
4. Download the report and load it in the local UI

In [ ]:
#@title 1. Setup (Run once) { display-mode: "form" }
import os, sys
REPO_DIR = '/content/baddyCoach'
BRANCH = 'master'

# Install Python dependencies
!pip install -q torch torchvision ultralytics onnxruntime-gpu opencv-python-headless scipy pyyaml gdown tqdm

# Clone the master branch (remote HEAD points to main, so -b is required)
if os.path.exists(REPO_DIR):
    !rm -rf "$REPO_DIR"
!git clone --depth 1 -b {BRANCH} https://github.com/sujkuttan/baddyCoach.git "$REPO_DIR"

os.chdir(REPO_DIR)
print(f'\nWorking directory: {os.getcwd()}')
print(f'Branch: {BRANCH}')
print(f'Contents: {os.listdir(".")}')

# Models will auto-download on first pipeline run via ensure_model()
print('\nSetup complete!')

In [ ]:
#@title 2. Upload Video { display-mode: "form" }
import os, gdown

VIDEO_ID = '1aA_3keNIfCBjkNC9isovHwTQjfVejQdC'
video_path = '/content/test_match.mp4'
if not os.path.exists(video_path):
    print('Downloading test video...')
    gdown.download(id=VIDEO_ID, output=video_path, quiet=False)

print(f'\nVideo ready: {os.path.getsize(video_path) / 1024 / 1024:.1f} MB')
print(f'   Path: {video_path}')

In [ ]:
#@title 3. Run Pipeline { display-mode: "form" }
#@markdown On GPU (T4), expect ~5-15 min for a 30-min video.
POSE_MODEL = 'hybrid' #@param ['rtmpose', 'mmpose', 'hybrid']
#@markdown - `rtmpose`: Fast (default). RTMPose-M ONNX.
#@markdown - `mmpose`: Accurate. HRNet-W32 ONNX.
#@markdown - `hybrid`: MMPose strokes (training-consistent) + RTMPose hits (better confidence).
SAMPLE_RATE = 0 #@param {type:'integer'}
#@markdown - `0`: Auto (10fps). `2`: Every 2nd frame (15fps). `1`: Every frame (30fps, slow).

sample_arg = f'--sample-rate {SAMPLE_RATE}' if SAMPLE_RATE > 0 else ''
import os
os.chdir('/content/baddyCoach')
!python -u colab/pipeline.py "{video_path}" --output /content/report.json --device cuda --pose-model {POSE_MODEL} {sample_arg} --log /content/pipeline.log

print('\nPipeline complete! Report: /content/report.json')
if os.path.exists('/content/pipeline.log'):
    print(f'\nFull log saved to: /content/pipeline.log')

In [ ]:
#@title 4. View Summary & Download Outputs { display-mode: "form" }
import json, os, zipfile
from pathlib import Path
from google.colab import files

report_path = '/content/report.json'
debug_dir = Path('/content/debug')
log_path = '/content/pipeline.log'
zip_path = '/content/bmca_outputs.zip'

if not os.path.exists(report_path):
    print('Report not found. Run Step 3 first.')
else:
    report = json.load(open(report_path))

    print('=' * 50)
    print('  MATCH ANALYSIS SUMMARY')
    print('=' * 50)
    print(f'  Rallies: {len(report.get("rallies", []))}')
    print(f'  Shots: {report.get("shot_count", 0)}')
    print(f'  Players: {len(report.get("footwork", {}))}')

    sd = report.get('shot_distribution', {})
    if sd:
        print(f'\n  Top shots:')
        for s, p in sorted(sd.items(), key=lambda x: -x[1])[:5]:
            print(f'    {s:15s} {p*100:5.1f}%')

    if report.get('strengths'):
        print(f'\n  Strengths:')
        for s in report['strengths'][:3]: print(f'    - {s[:70]}')
    if report.get('weaknesses'):
        print(f'\n  Improve:')
        for w in report['weaknesses'][:3]: print(f'    - {w[:70]}')

    print('\n' + '=' * 50)
    print('  PACKAGING OUTPUTS')
    print('=' * 50)

    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        zf.write(report_path, 'report.json')
        print(f'  + report.json')

        if debug_dir.exists():
            for f in sorted(debug_dir.iterdir()):
                if ':Zone.Identifier' in f.name:
                    continue
                zf.write(f, f'debug/{f.name}')
                print(f'  + debug/{f.name}')

        if os.path.exists(log_path):
            zf.write(log_path, 'pipeline.log')
            print(f'  + pipeline.log')

    size_mb = os.path.getsize(zip_path) / 1024 / 1024
    print(f'\n  Zip: {zip_path} ({size_mb:.1f} MB)')
    print('\n' + '=' * 50)

    try:
        files.download(zip_path)
        print('Downloaded! Unzip locally, load report.json in BMCA UI.')
    except Exception as e:
        print(f'Zip download failed ({e}). Downloading individually...')
        files.download(report_path)
        if debug_dir.exists():
            for f in sorted(debug_dir.iterdir()):
                if ':Zone.Identifier' in f.name:
                    continue
                files.download(str(f))
        if os.path.exists(log_path):
            files.download(log_path)